# Kaggle Ensemble + Submission

Upload prediction files from Colab training as a **Kaggle Dataset**, then run this notebook.

## Instructions
1. After Colab training, download `v29_predictions/` from Google Drive to your computer
2. In this Kaggle notebook, click **Add Data** → **Upload** → select all prediction `.pkl` files
3. They appear under `/kaggle/input/` as a dataset named e.g. `v29-predictions`
4. Run all cells below

In [ ]:
# Cell 1: Install deps (minimal — ensemble only)
import sys, subprocess
def pip_install(pkg):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], capture_output=True)
for pkg in ["scikit-learn", "pandas", "numpy", "pyyaml", "xgboost"]:
    if subprocess.run([sys.executable, "-c", f"import {pkg.replace('-', '_')}"],
                      capture_output=True).returncode != 0:
        pip_install(pkg)
print("Deps ready")

In [ ]:
# Cell 2: Clone repo for ensemble code (light: only weight_optimizer + build_ensemble)
import os, shutil
from pathlib import Path

WORK_DIR = Path("/kaggle/working")
os.chdir(str(WORK_DIR))

REPO_URL = "https://github.com/NotShubham1112/Poly.git"
subprocess.run(["git", "clone", "--depth", "1", REPO_URL],
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

SRC = WORK_DIR / "Poly" / "polymer_competition"
for item in os.listdir(str(SRC)):
    s = SRC / item
    d = WORK_DIR / item
    if d.exists():
        shutil.rmtree(str(d)) if d.is_dir() else os.remove(str(d))
    shutil.copytree(str(s), str(d)) if s.is_dir() else shutil.copy2(str(s), str(d))

print("Code ready")

In [ ]:
# Cell 3: Symlink uploaded predictions to predictions/ dir
# Change DATASET_NAME to match what Kaggle shows after upload
DATASET_NAME = "v29-predictions"

import glob

pred_dir = WORK_DIR / "predictions"
pred_dir.mkdir(exist_ok=True)

# Find uploaded files
uploaded = list(Path("/kaggle/input").glob(f"{DATASET_NAME}/*.pkl"))
if not uploaded:
    # Try broader search
    uploaded = list(Path("/kaggle/input").rglob("*.pkl"))

for src in uploaded:
    dst = pred_dir / src.name
    if not dst.exists():
        shutil.copy2(str(src), str(dst))
print(f"Linked {len(uploaded)} prediction files from /kaggle/input/{DATASET_NAME}/")

In [ ]:
# Cell 4: Config setup
import yaml
with open("config.yaml") as f:
    cfg = yaml.safe_load(f)
cfg["experiment"]["version"] = "v29"
with open("config.yaml", "w") as f:
    yaml.dump(cfg, f, default_flow_style=False)
print(f"Version: v29, pred_dir: {cfg['paths']['predictions_dir']}")

In [ ]:
# Cell 5: Build weighted ensemble per target
!python -m ensemble.build_ensemble --config config.yaml --target tg
!python -m ensemble.build_ensemble --config config.yaml --target egc
print("Weighted ensembles built!")

In [ ]:
# Cell 6: Level-2 stacking per target (optional — skip if NaN issues)
!python -m ensemble.stacking_ensemble --config config.yaml --target tg 2>/dev/null || echo "Stacking tg skipped"
!python -m ensemble.stacking_ensemble --config config.yaml --target egc 2>/dev/null || echo "Stacking egc skipped"
print("Stacking done (or skipped)")

In [ ]:
# Cell 7: Generate combined submission (auto-skip NaN)
!python run_submission.py --target all 2>&1 || echo "run_submission.py had issues — trying manual blend..."

# If run_submission.py fails, do a manual ensemble from the weighted ensemble outputs
import pandas as pd
from pathlib import Path

sub_dir = Path("outputs/submissions")
final_csv = sub_dir / "submission.csv"

if not final_csv.exists() or pd.read_csv(final_csv)["target"].isna().any():
    print("Manual fallback: blending per-target CSVs...")
    tg_csv = sub_dir / "tg_preds.csv"
    egc_csv = sub_dir / "egc_preds.csv"
    if tg_csv.exists() and egc_csv.exists():
        tg = pd.read_csv(tg_csv)
        egc = pd.read_csv(egc_csv)
        sub = pd.concat([tg, egc]).sort_values("id").reset_index(drop=True)
        sub.to_csv(final_csv, index=False)
        print(f"Manual submission: {final_csv} ({len(sub)} rows)")

# Verify
sub = pd.read_csv(final_csv)
n_nan = int(sub["target"].isna().sum())
print(f"Verification: {len(sub)} rows, {n_nan} NaN, "
      f"ids=[{sub['id'].min()}, {sub['id'].max()}]")